# **Question 5.3.1**
**What happens if the input dimensions for the first layer are specified, but the dimensions for subsequent layers are not? Is initialisation carried out immediately?**<br><br>
It operates normally. The first layer is initialised immediately, but the other layers are not initialised until data passes through the model for the first time.

In [1]:
import torch
from torch import nn

"""Deferred Initialization"""
net = nn.Sequential(nn.LazyLinear(256), nn.ReLU(), nn.LazyLinear(10))
# 尚未初始化
print(net)

X = torch.rand(2, 20)
net(X)
print(net)

Sequential(
  (0): LazyLinear(in_features=0, out_features=256, bias=True)
  (1): ReLU()
  (2): LazyLinear(in_features=0, out_features=10, bias=True)
)
Sequential(
  (0): Linear(in_features=20, out_features=256, bias=True)
  (1): ReLU()
  (2): Linear(in_features=256, out_features=10, bias=True)
)


# **Question 5.3.2**
**What happens if an incompatible dimension is specified?**<br><br>
An error will be reported due to a mismatch in the dimensions of the matrices.

In [2]:
import torch

net = nn.Sequential(
    nn.Linear(20, 256), nn.ReLU(),
    nn.LazyLinear(128), nn.ReLU(),
    nn.LazyLinear(10))

X = torch.rand(2, 10)
try:
    net(X)
except Exception as e:
    print(e)

mat1 and mat2 shapes cannot be multiplied (2x10 and 20x256)


# **Question 5.3.3**
**What should be done if the input has different dimensions? Hint: Refer to the relevant section on parameter binding.**<br><br>
Add an additional linear layer and bind the weights of the first linear layer to those of this layer. This resolves the issue of dimensional mismatch whilst keeping the model’s weights unchanged. Note that in the code above, we assume the bias term of the first linear layer is zero, so there is no need to bind its parameters.

In [3]:
import torch
import torch.nn as nn

net = nn.Sequential(
    nn.Linear(20, 256), nn.ReLU(),
    nn.Linear(256, 128), nn.ReLU(),
    nn.Linear(128, 10))

# 添加额外的线性层
extra_layer = nn.Linear(10, 256)

# 将第一个线性层与额外的线性层的权重进行绑定
net[0].weight = extra_layer.weight

# 使用新的输入（维度为20）调用模型
X = torch.rand(2, 10)
net(X)

tensor([[ 0.1049, -0.0787,  0.1161, -0.0644,  0.0935,  0.1172, -0.0242, -0.0098,
          0.1112,  0.0423],
        [ 0.1255, -0.0684,  0.1137, -0.0918,  0.1369,  0.2136, -0.0123,  0.0085,
          0.0476,  0.0383]], grad_fn=<AddmmBackward0>)